In [1]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import multilabel_confusion_matrix
import numpy as np
import pandas as pd
import os
import json
import random
import argparse

# General Functions

In [2]:
import os
import json
import random 
import argparse


def read_json(path):
    with open(path, 'r') as file:
        data = json.load(file)
    return data

def write_json(data, path):
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    with open(path, 'w') as file:
        json.dump(data, file, indent=4)

In [20]:

import simplemma
from simplemma import text_lemmatizer
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def canonicalize(s: str) -> str:
    if s == "":
        return s
    s = s.strip().lower()
    s = lemmatizer.lemmatize(s, pos=wordnet.VERB)
    if s.endswith("ies"):           # e.g., "subsidiaries" -> "subsidiary"
        # print(s)
        return s[:-3] + "y"
    if s.endswith("s") and not s.endswith("ss"):
        # print(s)
        return s[:-1]               # crude plural -> singular
    return s


[nltk_data] Downloading package wordnet to /Users/sefika/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/sefika/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [21]:
def get_seen_gt(run_id, task_id, folder_path):
    relation_list = []
    for id in range(1, task_id+1):
        # print(id)
        file = folder_path+"/run{0}".format(run_id)+"/task{0}".format(id)+"/test.json"
        relations = [canonicalize(item['relation']) for item in read_json(file)]
        # print(len(relations))
        relation_list.extend(relations)
    return relation_list

In [22]:
def get_false_prediction(preds, gt):
    false_preds = []
    # print(f"pred len:{len(preds)}--- gt len:{len(gt)}")
    for id, pred in enumerate(preds):
        if pred != gt[id]:
            false = {'truth':gt[id], 'pred':pred}
            false_preds.append(false)
    
    return false_preds
        

In [23]:
def return_hallucinated_prediction(preds, predefined_relations):
    hallucinated = []
    predefined_relations = list(set(predefined_relations))
    predefined_relations = [ canonicalize(item.split(':')[-1]) for item in predefined_relations]
    # print(len(predefined_relations))
    preds = [canonicalize(pred.split(':')[-1]) for pred in preds]
    
    for id, pred in enumerate(preds):
        if pred in predefined_relations:
            continue
        hallucinated.append(pred)

    return hallucinated
            

In [24]:
def seen_predicts(result_folder, gt_folder, relation_folder):
    false_predictions = []
    hallucination = []
    for run_id in range(1,6):
        unique_hallucination = []
        for task_id in range(1,9):
            # print(task_id)
            file_path =  result_folder + "/model_{0}/".format(run_id) + "test_pred_{0}.json".format(task_id)
            preds = [canonicalize(pred['predict']) for pred in read_json(file_path)]
            gt_relations = get_seen_gt(run_id, task_id, gt_folder)
       
            false_preds = get_false_prediction(preds, gt_relations)
            
            hallucinated = return_hallucinated_prediction(preds, gt_relations)
            falses = {'run_id':run_id, "task_id":task_id, "false":false_preds, 'hallucination':hallucinated, 
                      'count_unique_hallucination': len(list(set(hallucinated))), 'unique_hallucinated_relations':list(set(hallucinated))}
            false_predictions.append(falses)
            unique_hallucination.extend(list(set(hallucinated)))
        hallucination.append({'run_id':run_id, 'unique_hallucination':list(set(unique_hallucination))})
    return false_predictions, hallucination
        

# TACRED

In [15]:
experiments= {
        # 'mas_discovery':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/mas_flan',
#              'mas':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/mas_correct_tacred',
#              'synaptic_intelligenece':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/si_tacred_flan_random',
#              'elastic_weight_consolidation':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/ewc_corrected_tacred_100',
        #       'baseline':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/baseline',
              "ojas":'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/tacred/tacred_oja'
             }

In [16]:
ground_truth = {'test':'/Users/sefika/phd_projects/llm-catastrophic-re/FSCRE/tacred/5way5shot-test'}
relations = {'relation': '/Users/sefika/phd_projects/llm-catastrophic-re/FSCRE/tacred/relations/'}

In [17]:
for key, value in experiments.items():
    false_predictions, hallucination = seen_predicts(experiments[key], ground_truth['test'], relations['relation'])
    false_pred_path = value+"/"+key+'_false_pred.json'
    hallucination_path = value+"/"+key+'_hallucination_pred.json'
    write_json(false_predictions, false_pred_path)
    write_json(hallucination, hallucination_path)

In [171]:
## Hallucinated predictions

## FewRel

In [25]:
# experiments= {'mas_discovery':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/fewrel/mas_fewrel',
#              'mas':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/fewrel/mas_correct_fewrel',
#              'synaptic_intelligenece':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/fewrel/si_fewrel',
#              'elastic_weight_consolidation':'/Users/sefika/phd_projects/llm-catastrophic-re/last_results/fewrel/ewc_corrected_fewrel_100'
#              }
experiments= {'baseline':'/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/baseline',
             'mas':'/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas',
             'synaptic_intelligenece':'/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/si',
             'elastic_weight_consolidation':'/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/ewc'
             }

In [26]:
ground_truth = {'test':'/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/fewrel/test'}
relations = {'relation': '/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/fewrel/relations'}

In [27]:
for key, value in experiments.items():
    false_predictions, hallucination = seen_predicts(experiments[key], ground_truth['test'], relations['relation'])
    false_pred_path = value+"/"+key+'_canonical_false_pred.json'
    hallucination_path = value+"/"+key+'_canonical_hallucination_pred.json'
    print(f"Writing false predictions to {false_pred_path}")
    write_json(false_predictions, false_pred_path)

    write_json(hallucination, hallucination_path)

Writing false predictions to /Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/baseline/baseline_canonical_false_pred.json
Writing false predictions to /Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas/mas_canonical_false_pred.json
Writing false predictions to /Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/si/synaptic_intelligenece_canonical_false_pred.json
Writing false predictions to /Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/ewc/elastic_weight_consolidation_canonical_false_pred.json
